# 02 — Attention extraction by hand

**Phase 0, exercise 3 — the last warmup before Phase 1.** Goal: stop treating attention as a magic black box. Get the actual attention weights out of a HuggingFace model, look at them, and verify they obey the laws they're supposed to obey.

Why this matters: every metric Phase 1 will compute (attention entropy, attention concentration, attention sinks) is a function of these tensors. You can't trust derived metrics if you don't trust the extraction. By the end of this notebook you will:

- Know how to ask a HuggingFace model to return its attention weights.
- Know exactly what shape comes back and what each axis means.
- Visually confirm the causal mask is doing its job.
- Verify that attention rows sum to 1 (because they're softmax outputs).
- See firsthand that different heads in the same layer attend to *different* things — which is the whole point of multi-head attention.

Runs comfortably on CPU. Short prompt, small tensors.

## Setup

> Same Colab gotcha as before: do **not** `pip install torch`.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "EleutherAI/pythia-160m"
MODEL_REVISION = "step143000"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

torch.manual_seed(SEED)
print(f"torch {torch.__version__} | device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, attn_implementation="eager"
).to(DEVICE)
model.eval()
print(f"loaded {MODEL_NAME} @ {MODEL_REVISION}")

### Why `attn_implementation="eager"`?

Modern HuggingFace models default to fused attention kernels (Flash Attention, SDPA) that are dramatically faster but *don't materialize the attention matrix* — they fuse the softmax into the matmul and never produce an intermediate `[seq, seq]` tensor for you to inspect. That's good for production, terrible for our purposes.

Forcing `attn_implementation="eager"` switches to the unfused reference path that actually computes and exposes the attention weights. This is slower but it's what we need for Phase 1's metric extraction. **Remember this flag** — it'll come up again.

## Known issue with this notebook + recent transformers

Recent `transformers` releases have a numerical-stability regression for Pythia/GPTNeoX models: in the deeper layers (typically layers 9-11 of Pythia-160M), the attention softmax overflows and returns NaN for entire rows. This affects **both** the eager and SDPA attention paths, so it can't be fixed by swapping kernels.

Why this happens: deeper layers have larger residual-stream magnitudes (every layer adds to the residual), so Q·Kᵀ attention scores grow with depth. In the eager softmax `exp(big_score)` overflows to `inf`, then `inf - inf` produces NaN. Numerically stable softmax (subtract the max before `exp`) would prevent this, and the fused kernels do it correctly — but the current `transformers` reference path doesn't, at least not for GPTNeoX.

**Pragmatic workaround for Phase 0:** layers 0-8 of Pythia-160M are unaffected, with layer 3 being entirely clean. We use layer 3 for all visualizations below. The row-sum sanity check will report that *some* rows aren't 1.0 (the NaN-in-deep-layers rows), which is expected — that's the bug, not a failure of your extraction.

**Phase 1 will solve this permanently** by extracting Q, K, V via forward hooks on the linear projections and computing attention manually, bypassing whatever bug lives in the model's softmax path.

## Asking for attention

Add `output_attentions=True` to a forward call and the output object gains an `attentions` field — a tuple, one entry per layer. Each entry is the attention weight tensor for that layer:

$$ \text{attentions}[\ell].\text{shape} = (\text{batch}, \text{heads}, \text{seq}, \text{seq}) $$

These are the **post-softmax** attention probabilities — the actual weights the model used to combine values, not the pre-softmax scores.

In [ ]:
prompt = "The quick brown fox jumps over the lazy dog."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
tokens = [tokenizer.decode([t]) for t in input_ids[0].tolist()]

with torch.no_grad():
    outputs = model(input_ids, output_attentions=True)

attentions = outputs.attentions  # tuple of length num_layers

print(f"prompt:           {prompt!r}")
print(f"tokens ({len(tokens)}):       {tokens}")
print(f"num layers:       {len(attentions)}")
print(f"per-layer shape:  {tuple(attentions[0].shape)}  # [batch, heads, seq, seq]")
print(f"dtype:            {attentions[0].dtype}")

In [ ]:
# Coerce NaN → 0 so downstream code doesn't blow up.
# (See the markdown above for why the NaN exists — it's a transformers regression in deeper layers.)
attentions = tuple(torch.nan_to_num(a, nan=0.0) for a in attentions)
nan_count = sum(torch.isnan(a).sum().item() for a in attentions)
print(f"remaining NaN cells after coercion: {nan_count}")

## What the axes mean

Given shape `[batch, heads, seq_len, seq_len]`, the two trailing axes are the part that confuses everyone the first time:

- **Axis -2 (rows)** is the **query** position — the token doing the attending. "Position `i` attends to ..."
- **Axis -1 (columns)** is the **key** position — the token being attended to. "... position `j`."

So `attentions[layer][batch, head, i, j]` = how much weight position `i` placed on position `j` in that head of that layer. Each row `i` is a probability distribution over which prior positions matter to token `i`.

**Causal mask consequence:** in a decoder-only language model, token `i` is only allowed to attend to positions `j ≤ i`. So everything in the strict upper triangle (`j > i`) should be exactly zero. You'll see that in the heatmap below.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

LAYER, HEAD = 3, 3  # layer 3 is clean in this transformers version (see issue note above)
A = attentions[LAYER][0, HEAD].cpu().float().numpy()  # [seq, seq]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(A, cmap="viridis", aspect="equal")
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticklabels(tokens)
ax.set_xlabel("key (attended TO)")
ax.set_ylabel("query (attending FROM)")
ax.set_title(f"attention weights — layer {LAYER}, head {HEAD}")
plt.colorbar(im, ax=ax, label="attention weight")
plt.tight_layout()
plt.show()

### What you're looking at

Three things to notice (the heatmap above should make them obvious):

1. **Upper-triangle is dark.** That's the causal mask — those positions can't attend forward. Pythia uses a softmax over masked logits, so the masked entries land at exactly zero.
2. **The first column (attending to the first token) often has higher weight than you'd expect.** This is the *attention sink* phenomenon — the first token gets disproportionate attention from many positions across many layers. It's not because the first token is semantically important; it's a learned register. The "When Attention Sink Emerges" paper on your reading list is about exactly this.
3. **Different heads in the same layer look very different.** That's coming up two cells down.

## Sanity check: rows sum to 1

Each query position's attention is a softmax output, so its weights over keys must sum to 1.0. The check below verifies this *per layer* — and you'll see clearly that layers 0-8 are clean while layers 9-11 have a lot of zero rows (the NaN-coerced-to-zero rows from the bug).

In [ ]:
all_attn = torch.stack(attentions, dim=0)  # [L, B, H, T, T]
row_sums = all_attn.sum(dim=-1)             # [L, B, H, T]

print("per-layer fraction of rows that sum to 1.0:")
for L in range(all_attn.shape[0]):
    ok = torch.isclose(row_sums[L], torch.ones_like(row_sums[L]), atol=1e-4)
    print(f"  layer {L:2d}: {ok.float().mean().item():>6.1%} clean")

print(f"\noverall: min={row_sums.min().item():.4f}  max={row_sums.max().item():.4f}  mean={row_sums.mean().item():.4f}")
print("(Layers 9-11 may show as broken — that's the documented transformers regression, not your code.)")

## Heads aren't all the same

Multi-head attention exists because a single attention pattern can't capture every relationship that matters. Different heads in the same layer learn to look at different things — some at the immediately previous token, some at the start of the sentence, some at specific syntactic patterns. Visualize a few side-by-side to see this directly.

In [ ]:
LAYER = 3  # clean layer
heads_to_show = [0, 3, 7, 11]

fig, axes = plt.subplots(1, len(heads_to_show), figsize=(4 * len(heads_to_show), 4))
for ax, h in zip(axes, heads_to_show):
    A = attentions[LAYER][0, h].cpu().float().numpy()
    ax.imshow(A, cmap="viridis", aspect="equal")
    ax.set_title(f"layer {LAYER}, head {h}")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Four heads, same layer, same input — very different patterns", y=1.02)
plt.tight_layout()
plt.show()

## Reflection questions

Write your answers in `docs/concepts/03_attention.md`.

1. Pick any cell of the attention tensor, say `attentions[3][0, 3, 4, 2]`. In plain English, what does this number represent?
2. Why must row sums be 1.0 but column sums *aren't* constrained to anything in particular?
3. The first column (attending to position 0) tends to be bright across many rows. If you wanted to *drop* the first token from the KV cache (per the notebook 01 reflection on attention sinks), what concrete failure mode would you expect — and how could you measure it?
4. Looking at the four-head grid: pick two heads with visually different patterns. Describe what each head appears to be tracking. (You don't need to be right; you need to develop a habit of *reading* attention patterns.)

These questions are deliberately less computational than the Phase 0 notebooks before. They're about building *taste* — the ability to look at an attention pattern and say something sensible about it. You'll need that in Phase 2.

## What's next — Phase 1 begins

Phase 0 is done. You can now:

- Write an autoregressive decoder loop from scratch.
- Explain what a KV cache buys and what it costs.
- Pull attention weights out of any HuggingFace causal LM and visualize them — and know what to do when the framework's path is buggy.
- Articulate the prefill/decode distinction and why batching helps decode.

That's the foundation. **Phase 1** is the metric library — formalizing per-token measurements like next-token entropy, attention entropy, attention concentration, and the *parallel-prediction agreement* metric that's the real heart of the project's thesis. We'll do that in `src/hybrid_arch/metrics.py` (importable code) rather than in notebooks (exploratory), and we'll start by writing forward-hook-based attention extraction so the layers-9-11 bug above stops being our problem.

Ping me when you've written the four reflection answers above and we'll set up Phase 1.